In [46]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import mesa
import pandas as pd

from mesa.visualization import SolaraViz, make_plot_component, make_space_component


In [51]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import mesa


def update_expectation(expected_outcome, outcome, alpha):
    expected_outcome += alpha * (outcome - expected_outcome)
    return expected_outcome




class AdoptionAgent(mesa.Agent):
    """An agent with skill, and the ability to use collective intelligence."""

    def __init__(self, model):
        """initialize an AdoptionAgent instance.

        Args:
            model: A model instance
        """
        super().__init__(model)
        
        "Preset starting variables for the model:"
        self.skill = np.clip(np.random.normal(loc=0.5, scale=0.2), 0, 1)
        self.ability = 0
        self.wins = 1
        self.losses = 1
        self.aichoice = 0
        self.expected = 0.5
        self.outcome = 0

        "Variables that can be changed between batches:"
        self.alpha = 0.1
        self.beta = 0.05
        self.boost = 0.10
        self.middle = 0.10
        self.scaler = 0.02
        


    def move(self):
        """move to a random neighboring cell."""
        possible_steps = self.model.grid.get_neighborhood(
            self.pos, moore=True, include_center=False
        )
        new_position = self.random.choice(possible_steps)
        self.model.grid.move_agent(self, new_position)

    def compete(self):
        """One agent picks another agent randomly and compares scores updating all info"""
        cellmates = self.model.grid.get_cell_list_contents([self.pos])
        if len(cellmates) > 1:
            other = self.random.choice(cellmates)

            self.expected = update_expectation(self.expected, self.outcome, self.alpha)
            self.aichoice = np.random.choice([0, 1], p=[1 - self.expected, self.expected])

            other.expected = update_expectation(other.expected, other.outcome, other.alpha)
            other.aichoice = np.random.choice([0, 1], p=[1 - other.expected, other.expected])

            self.ability = self.skill + self.aichoice * self.boost + np.clip(np.random.normal(loc= self.middle , scale= self.scaler ), 0, 1)
            other.ability = other.skill + other.aichoice * boost + np.clip(np.random.normal(loc= other.middle , scale= other.scaler ), 0, 1)
            self.skill = np.clip(self.skill + ((-1)**self.aichoice ) * self.beta, 0, 1)
            if self.ability > other.ability:
                self.wins += 1
                self.outcome = 1 * self.aichoice

            else:
                self.losses += 1
                self.outcome = 1 - self.aichoice




    def step(self):
        """do one step of the agent."""
        self.move()
        self.compete()


class AdoptionModel(mesa.Model):
    """A model with some number of agents."""

    def __init__(self,
                 alpha = 0.1,
                 beta = 0.05,
                 boost = 0.10,
                 middle = 0.10,
                 scaler = 0.02,
                 n=10,
                 width=10,
                 height=10,
                 seed=None):
        
        """Initialize a competition instance.

        Args:
            N: The number of agents.
            width: width of the grid.
            height: Height of the grid.
        """
        super().__init__(seed=seed)
        self.num_agents = n
        self.alpha = alpha
        self.beta = beta
        self.boost = boost
        self.middle = middle
        self.scaler = scaler
        self.grid = mesa.space.MultiGrid(width, height, True)

        # Create agents
        agents = AdoptionAgent.create_agents(model=self, n=n)
        # Create x and y positions for agents
        x = self.rng.integers(0, self.grid.width, size=(n,))
        y = self.rng.integers(0, self.grid.height, size=(n,))
        for a, i, j in zip(agents, x, y):
            # Add the agent to a random grid cell
            self.grid.place_agent(a, (i, j))

        self.datacollector = mesa.DataCollector(
            agent_reporters={
                "Skill": "skill",
                "Ability": "ability",
                "Wins": "wins",
                "Losses": "losses",
                "AI Usage Choice": "aichoice",
                "Expected": "expected",
                "Outcome": "outcome"
            }
        )
        self.datacollector.collect(self)

    def step(self):
        """do one step of the model"""
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)

In [52]:
model = AdoptionModel(100)
for _ in range(200):
    model.step()


data = model.datacollector.get_agent_vars_dataframe()
# Use seaborn

data


Skill   Ability  Wins  Losses  AI Usage Choice  Expected  \
Step AgentID                                                                
0    1        0.226840  0.000000     1       1                0  0.500000   
     2        0.247288  0.000000     1       1                0  0.500000   
     3        0.597680  0.000000     1       1                0  0.500000   
     4        0.627685  0.000000     1       1                0  0.500000   
     5        0.876263  0.000000     1       1                0  0.500000   
...                ...       ...   ...     ...              ...       ...   
200  6        0.330788  0.344794     1      12                0  0.486959   
     7        0.299909  0.353844     1      28                0  0.490564   
     8        0.480044  0.717110     2      11                1  0.437729   
     9        0.900000  0.920811     6      11                0  0.579969   
     10       0.430899  0.692800     5      15                1  0.619560   

              Outcome  
Step AgentID           
0    1              0  
     2              0  
     3              0  
     4              0  
     5              0  
...               ...  
200  6              1  
     7              1  
     8              0  
     9              1  
     10             1  

[2010 rows x 7 columns]

In [53]:
params = {"width": 10,
          "height": 10,
          "alpha": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
          "beta": 0.05,
          "boost": 0.05,
          "middle": 0.05,
          "scaler": 0.05,
          }

results = mesa.batch_run(
    AdoptionModel,
    parameters=params,
    iterations=5,
    max_steps=100,
    number_processes=1,
    data_collection_period=1,
    display_progress=True,
)


results_df = pd.DataFrame(results)
print(results_df.keys())

  0%|          | 0/30 [00:00<?, ?it/s]

Index(['RunId', 'iteration', 'Step', 'width', 'height', 'alpha', 'beta',
       'boost', 'middle', 'scaler', 'AgentID', 'Skill', 'Ability', 'Wins',
       'Losses', 'AI Usage Choice', 'Expected', 'Outcome'],
      dtype='object')


In [37]:
results_df

,RunId,iteration,Step,width,height,alpha,AgentID,Skill,Ability,Wins,Losses,AI Usage Choice,Expected,Outcome
0,0,0,0,10,10,0.0,1,0.692617,0.000000,1,1,0,0.500000,0
1,0,0,0,10,10,0.0,2,0.626150,0.000000,1,1,0,0.500000,0
2,0,0,0,10,10,0.0,3,0.668454,0.000000,1,1,0,0.500000,0
3,0,0,0,10,10,0.0,4,0.347852,0.000000,1,1,0,0.500000,0
4,0,0,0,10,10,0.0,5,0.376714,0.000000,1,1,0,0.500000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30295,29,4,100,10,10,0.5,6,0.720089,0.774646,6,3,0,0.528116,0
30296,29,4,100,10,10,0.5,7,0.090417,0.333071,2,9,1,0.439665,0
30297,29,4,100,10,10,0.5,8,0.223659,0.323463,1,6,0,0.536216,1
30298,29,4,100,10,10,0.5,9,0.520167,0.584975,4,4,0,0.583849,0
